# CulSen-BN: Master Training & Evaluation Notebook for Google Colab

This notebook provides the complete executable pipeline to run Transformer baselines (BanglaBERT & XLM-R) and the full **CulSen-BN** uncertainty-aware, culture-sensitive contrastive framework on Google Colab (GPU).

In [ ]:
# 1. GPU & Environment Verification
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Clone Repository & Install Dependencies
!git clone https://github.com/ragibcs/coke_studio_bangla_sentiment.git || echo "Already cloned"
%cd coke_studio_bangla_sentiment
!pip install -q -e . transformers datasets scikit-learn pyyaml pandas tqdm matplotlib seaborn

In [ ]:
# 3. Verify Dataset Audit and Leakage-Free Splits
!python scripts/audit_dataset.py --config configs/data.yaml
!python scripts/create_splits.py --config configs/data.yaml

In [ ]:
# 4. Train Classical SVM Baseline (CPU-efficient)
!python scripts/train_svm.py --config configs/baseline_svm.yaml

In [ ]:
# 5. Train Transformer Baselines (BanglaBERT & XLM-R on GPU)
from culsen_bn.training.train_transformer import train_transformer_baseline

print("=== Training BanglaBERT Baseline ===")
banglabert_results = train_transformer_baseline("configs/baseline_transformer.yaml", model_override="banglabert")

print("=== Training XLM-RoBERTa Baseline ===")
xlmr_results = train_transformer_baseline("configs/baseline_transformer.yaml", model_override="xlm_roberta")

In [ ]:
# 6. Train Full CulSen-BN Framework (Variant B5)
from culsen_bn.training.train_culsen import train_culsen_model

print("=== Training CulSen-BN Full Framework (B5) ===")
culsen_results = train_culsen_model(
    experiment_config="configs/experiment.yaml",
    variant_name="B5_full",
    pretrained_model_name="sagorsarker/banglabert"
)

In [ ]:
# 7. Run Protocol B Cross-Season Generalization Evaluation
!python scripts/train_cross_season_svm.py

In [ ]:
# 8. Export Final Tables and Summary Reports
!python scripts/export_results.py